# Is this email critical for our project?


A customer writes to us. The agent has to find the project the email belongs to, compare it with the specification, offer and contract, and apply the business rules of the ontology (email criticality, escalation, change requests).

**Prerequisites:** `./deploy.sh` was run, `az login` is active and the Fabric capacity is running.

## 1. Connect to the demo


In [1]:
# All cells share one Bash process, so variables and functions carry over.
RG=rg-projplan-demo
AGENT=project-planning-agent
output() { az deployment group show -g "$RG" -n main --query "properties.outputs.$1.value" -o tsv; }
PROJECT_ENDPOINT=$(output foundryProjectEndpoint)
CAPACITY=$(output fabricCapacityName)
echo "Project endpoint: $PROJECT_ENDPOINT"
echo "Agent:            $AGENT"
echo "Capacity state:   $(az resource show -g "$RG" -n "$CAPACITY" --resource-type Microsoft.Fabric/capacities --query properties.state -o tsv)"

Project endpoint: https://ai-projplan-axowf3.services.ai.azure.com/api/projects/project-planning
Agent:            project-planning-agent
Capacity state:   Active


## 2. Helper functions
`ask` sends a question to the agent through the Foundry Responses API. `triage` wraps an incoming email into a question. `tools_used` shows which tool calls the agent made.


In [2]:
ask() {
  local token body
  token=$(az account get-access-token --scope https://ai.azure.com/.default --query accessToken -o tsv)
  body=$(jq -n --arg agent "$AGENT" --arg input "$1" '{agent_reference: {type: "agent_reference", name: $agent}, input: $input}')
  RESPONSE=$(curl -s -X POST "$PROJECT_ENDPOINT/openai/v1/responses" \
    -H "Authorization: Bearer $token" -H "Content-Type: application/json" -d "$body")
  if [ "$(echo "$RESPONSE" | jq '.error != null')" = true ]; then
    echo "$RESPONSE" | jq -r '.error.message'
    return 1
  fi
  echo "$RESPONSE" | jq -r '[.output[] | select(.type == "message") | .content[] | select(.type == "output_text") | .text] | join("\n")'
}

triage() {
  ask "We just received this email. 1) Which project does it belong to? The sender may have written before. 2) Is it critical for the project? Start your answer with a clear verdict and give the reason. 3) What must happen next and who is responsible?

From: $1
Subject: $2
Body: $3"
}

tools_used() {
  echo "$RESPONSE" | jq -r '.output[] | select(.type == "mcp_call") | "\(.name): \(.arguments[0:220])"'
}

echo "Helpers ready"


Helpers ready


## 3. Email 1: a customer wants an earlier completion
The email does not name the project. The agent has to identify it from the sender.


In [3]:
FROM="client@nordbau.example"
SUBJECT="Earlier move-in"
BODY="Our tenant wants to move in on 2027-11-01 instead of January. Please bring the completion date forward by six weeks."
echo "From:    $FROM"
echo "Subject: $SUBJECT"
echo
echo "$BODY"


From:    client@nordbau.example
Subject: Earlier move-in

Our tenant wants to move in on 2027-11-01 instead of January. Please bring the completion date forward by six weeks.


In [4]:
triage "$FROM" "$SUBJECT" "$BODY"


Verdict:
1. The email from client@nordbau.example belongs to Project P-100 — Office Building Hamburg. This is based on sender history, as previous correspondence from this sender is always tied to that project .

2. The request is critical for the project. Moving the completion date forward by six weeks directly impacts the project timeline and deliverables—this is a significant change to scheduling and must be treated as critical .

3. Next steps: The email must be escalated to the project manager, Anna Becker (anna.becker@example.com), within 1 working day. Anna is responsible for reviewing the schedule impact and coordinating the follow-up response .


What did the agent do? The agent only uses the Fabric IQ tool. The ontology holds the data, the documents and the rules.


In [5]:
tools_used


ask_ontology: {"request":"1) Which project does the email from client@nordbau.example about earlier move-in belong to? Consider previous correspondence from this sender.\n2) Is this request critical for the project? Explain why.\n3) W


## 4. Email 2: a routine request
For contrast, an email that should not be critical.


In [6]:
FROM="team@isar.example"
SUBJECT="Site plan"
BODY="Could you send us the current site plan as a PDF?"
echo "From:    $FROM"
echo "Subject: $SUBJECT"
echo
echo "$BODY"


From:    team@isar.example
Subject: Site plan

Could you send us the current site plan as a PDF?


In [7]:
triage "$FROM" "$SUBJECT" "$BODY"


Verdict: No direct evidence found that the email about "Site plan" from team@isar.example belongs to any project in the records.

Reason:
- The only communication from team@isar.example in the system is about "Holiday schedule," not "Site plan."
- That "Holiday schedule" message is linked to project P-200 (Residential Complex Munich), but is not about site plans or urgent requests.
- There is no recorded task, log, or assignment connected to "Site plan" involving this sender.

Is it critical?
No, according to current records. The only documented communication from this sender relates to office closure dates and does not impact project milestones, scope, or contract terms.

What must happen next and who is responsible?
No action required based on the available records and project policies. If there is an undocumented request regarding the site plan, you should clarify with project manager Ben Fischer (ben.fischer@example.com) connected to P-200. Otherwise, follow standard procedures for

## Summary
* The agent prompt contains no rating scheme. The rules `EmailCriticality`, `EscalationPolicy` and `ChangeRequestPolicy` live in the ontology and are applied by the Fabric IQ tool.
* The project, its manager and the documents come from the ontology tables, so the answer cites the clauses of the specification, offer and contract.
